# Evaluation — Python Q&A Assistant

This notebook runs queries against the **live API** (gate → FAISS → rerank → Groq) and captures the real responses, so the outputs below are not hand-written.

**Read this first.** The committed demo index is a **small 150-doc slice** (top-150 highest-scored questions) so it builds and deploys in seconds. Those top questions are mostly *language* questions, so library topics like pandas are barely covered. The full 50k-pair corpus (same code, `INDEX_MAX_DOCS=0`) closes most of these gaps. I kept the small-index failures in on purpose — they show the system refusing honestly instead of bluffing.

`rel` (relevance) gate is at **0.66**. Below it, the question is refused before the LLM is ever called.

In [1]:
import os, time, textwrap

# Run from the project root so data/faiss_index resolves whether launched here or from notebooks/.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
assert os.path.exists("data/faiss_index"), f"index not found from {os.getcwd()}"

from fastapi.testclient import TestClient
from app.main import app

# TestClient as a context manager runs the lifespan: loads the index + models once.
client = TestClient(app)
client.__enter__()
print("health:", client.get("/health").json())

/Users/vikrantdhanwadia/Gaurav Burman/AI Project/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


15:50:09 INFO sentence_transformers.SentenceTransformer - Use pytorch device_name: mps


15:50:09 INFO sentence_transformers.SentenceTransformer - Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5


15:50:14 INFO faiss.loader - Loading faiss.


15:50:14 INFO faiss.loader - Successfully loaded faiss.


15:50:16 INFO sentence_transformers.cross_encoder.CrossEncoder - Use pytorch device: mps


15:50:17 INFO app - startup: index loaded, 380 vectors


15:50:17 INFO httpx - HTTP Request: GET http://testserver/health "HTTP/1.1 200 OK"


health: {'status': 'ok', 'model': 'llama-3.3-70b-versatile', 'embed_model': 'BAAI/bge-small-en-v1.5', 'index_loaded': True, 'num_vectors': 380}


In [2]:
def ask(question, show_chars=600):
    t0 = time.perf_counter()
    r = client.post("/ask", json={"question": question})
    wall = (time.perf_counter() - t0) * 1000
    print("Q:", question)
    print(f"HTTP {r.status_code}  |  wall {wall:.0f} ms")
    if r.status_code != 200:
        print("body:", r.json())
        print("-" * 80)
        return r
    body = r.json()
    ans = body["answer"].strip()
    print("A:", textwrap.shorten(ans, show_chars, placeholder=" ..."))
    srcs = body.get("sources", [])
    if srcs:
        print("sources:")
        for s in srcs:
            print(f"   - {s['title']}  ({s['url']})")
    else:
        print("sources: [] (refused / no grounding)")
    print("-" * 80)
    return r

## 8 content queries
A mix of clearly-in-scope Python questions, a coverage-gap case (pandas), borderline/out-of-scope, and a clearly non-Python query. The point is to show both good grounded answers **and** honest refusals.

In [3]:
queries = [
    "how do I reverse a list in python",
    "what is the difference between is and == in python",
    "list comprehension vs map in python",
    "decorators",
    "how do I drop rows with NaN in a pandas dataframe",
    "why do I get an IndentationError in python",
    "how do I deploy a django app to aws",
    "what is the capital of france",
]
for q in queries:
    ask(q)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

15:50:18 INFO httpx - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


15:50:18 INFO api - ask cache=miss ms=1036 grounded=True sources=4 q='how do I reverse a list in python'


15:50:18 INFO httpx - HTTP Request: POST http://testserver/ask "HTTP/1.1 200 OK"


Q: how do I reverse a list in python
HTTP 200  |  wall 1039 ms
A: ```python my_list = [1, 2, 3, 4, 5] reversed_list = my_list[::-1] print(reversed_list) # [5, 4, 3, 2, 1] ``` This uses the same extended slice syntax as reversing a string [1].
sources:
   - Reverse a string in Python  (https://stackoverflow.com/questions/931092)
   - Making a flat list out of list of lists in Python  (https://stackoverflow.com/questions/952914)
   - Remove items from a list while iterating in Python  (https://stackoverflow.com/questions/1207406)
   - How can I represent an 'Enum' in Python?  (https://stackoverflow.com/questions/36932)
--------------------------------------------------------------------------------


...

Q: why do I get an IndentationError in python
HTTP 200  |  wall 444 ms
A: I don't have a grounded answer. The provided context [1], [2], [3], [4] does not cover the question about IndentationError in Python.
sources:
   - Manually raising (throwing) an exception in Python  (https://stackoverflow.com/questions/2052390)
   - Is there a way to substring a string in Python?  (https://stackoverflow.com/questions/663171)
   - Difference between __str__ and __repr__ in Python  (https://stackoverflow.com/questions/1436703)
   - Does Python have a string contains substring method?  (https://stackoverflow.com/questions/3437059)
--------------------------------------------------------------------------------
15:50:22 INFO api - ask cache=miss ms=11 grounded=False sources=0 q='how do I deploy a django app to aws'


15:50:22 INFO httpx - HTTP Request: POST http://testserver/ask "HTTP/1.1 200 OK"


Q: how do I deploy a django app to aws
HTTP 200  |  wall 16 ms
A: I don't have a grounded answer for that in my Stack Overflow sources. Try rephrasing it as a Python question, or ask something closer to common data-science topics.
sources: [] (refused / no grounding)
--------------------------------------------------------------------------------
15:50:22 INFO api - ask cache=miss ms=10 grounded=False sources=0 q='what is the capital of france'


15:50:22 INFO httpx - HTTP Request: POST http://testserver/ask "HTTP/1.1 200 OK"


Q: what is the capital of france
HTTP 200  |  wall 12 ms
A: I don't have a grounded answer for that in my Stack Overflow sources. Try rephrasing it as a Python question, or ask something closer to common data-science topics.
sources: [] (refused / no grounding)
--------------------------------------------------------------------------------


## Query #9 — input validation (422)
Empty / whitespace input is rejected by pydantic (`min_length`) before it ever reaches the pipeline.

In [4]:
ask("   ")

15:50:22 INFO httpx - HTTP Request: POST http://testserver/ask "HTTP/1.1 422 Unprocessable Entity"


Q:    
HTTP 422  |  wall 2 ms
body: {'detail': [{'type': 'string_too_short', 'loc': ['body', 'question'], 'msg': 'String should have at least 3 characters', 'input': '   ', 'ctx': {'min_length': 3}}]}
--------------------------------------------------------------------------------


<Response [422 Unprocessable Entity]>

## Query #10 — cache hit
Repeating an earlier question is served from the TTL cache. Watch the wall-clock time drop versus the first call above.

In [5]:
ask("how do I reverse a list in python")  # repeat of #1 -> cached

15:50:22 INFO api - ask cache=hit ms=1036 q='how do I reverse a list in python'


15:50:22 INFO httpx - HTTP Request: POST http://testserver/ask "HTTP/1.1 200 OK"


Q: how do I reverse a list in python
HTTP 200  |  wall 1 ms
A: ```python my_list = [1, 2, 3, 4, 5] reversed_list = my_list[::-1] print(reversed_list) # [5, 4, 3, 2, 1] ``` This uses the same extended slice syntax as reversing a string [1].
sources:
   - Reverse a string in Python  (https://stackoverflow.com/questions/931092)
   - Making a flat list out of list of lists in Python  (https://stackoverflow.com/questions/952914)
   - Remove items from a list while iterating in Python  (https://stackoverflow.com/questions/1207406)
   - How can I represent an 'Enum' in Python?  (https://stackoverflow.com/questions/36932)
--------------------------------------------------------------------------------


<Response [200 OK]>

## Observations on quality

- **Grounding holds.** Off-topic questions (capital of France) are refused at the gate before any LLM call. Borderline ones that slip past the gate are still declined by the prompt when the retrieved context doesn't actually cover them — a two-layer honesty design. That refuse-rather-than-bluff behaviour is the bit worth scrutinising.
- **The honest weakness is coverage, not retrieval.** The cross-encoder orders candidates well; there just aren't pandas/library docs in a 150-question slice, so pandas gets refused. The fix is the index-size knob (`INDEX_MAX_DOCS`), not a pipeline change.
- **Citations are real.** Grounded answers cite the Stack Overflow questions they were built from, and the answers adapt those sources rather than copying them (e.g. reversing a list reuses the slice idiom from a reverse-a-string answer).
- **The gate at 0.66 is tuned to this small index.** bge-small has a high cosine floor (off-topic still scores ~0.5–0.6), so the threshold sits in the gap just above that. It should be re-tuned when the index size changes.
- **Cache pays off.** The repeated question returns from the TTL cache in ~1 ms versus ~1–2 s warm (or ~12 s cold), which is also the first lever for handling concurrency.

In [6]:
client.__exit__(None, None, None)  # run lifespan shutdown